# Cleaning And Transformations

### Creating Spark Session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CapstoneSession") \
    .config("fs.azure.account.key.<redacted>.blob.core.windows.net", "redacted") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()



### Connecting to cosmos DB

In [ ]:
endpoint = 'https://<redacted>.documents.azure.com:443/'
accountkey = '<redacted>'
database = 'auditdatabase'
container = 'auditcapstone'

df = spark.read.format("cosmos.oltp") \
    .option("spark.cosmos.accountEndpoint", endpoint) \
    .option("spark.cosmos.accountKey", accountkey) \
    .option("spark.cosmos.database", database) \
    .option("spark.cosmos.container", container) \
    .load()

In [ ]:
# cache the data for optimization

cached_df = df.cache()

In [ ]:
# check if the data is complete

cached_df.count()

### Data Cleaning

In [ ]:
# drop the id column added by cosmos
cached_df = cached_df.drop('id')

In [ ]:
# drop duplicate values in the dataframe
cached_df  = cached_df.dropDuplicates()

In [ ]:
# count after removing duplicates

cached_df.count()

In [ ]:
# reorder the columns as per the original document

selected_columns = [
    "number", "incident_state", "active", "reassignment_count", "reopen_count", "sys_mod_count", 
    "made_sla", "caller_id", "opened_by", "opened_at", "sys_created_by", "sys_created_at", 
    "sys_updated_by", "sys_updated_at", "contact_type", "location", "category", "subcategory", 
    "u_symptom", "cmdb_ci", "impact", "urgency", "priority", "assignment_group", "assigned_to", 
    "knowledge", "u_priority_confirmation", "notify", "problem_id", "rfc", "vendor", 
    "caused_by", "closed_code", "resolved_by", "resolved_at", "closed_at"
]

cached_df = cached_df.select(*selected_columns)


In [ ]:
# remove the unwanted row (added as a reference to cosmos)

from pyspark.sql.functions import to_timestamp, col
cached_df = cached_df.where((col('caused_by') != 'code 5'))


In [ ]:
# sort each incidents by the order of occurance
 
from pyspark.sql.functions import to_timestamp, col, expr

sort_order = {
    "Active": 1,
    "New": 2,
    "Awaiting User Info": 3,
    "Awaiting Evidence": 4,
    "Awaiting Vendor": 5,
    "Awaiting Problem": 6,
    "Resolved": 7,
    "Closed": 8,
    "-100": 9,
}

df_with_order = cached_df.withColumn("sort_order", expr(f"""
    CASE incident_state
        {' '.join([f'WHEN "{key}" THEN {value}' for key, value in sort_order.items()])}
        ELSE 99  -- Default value for unexpected states
    END
"""))

# Sort the DataFrame by 'number', 'sys_mod_at', and the new 'sort_order'
sorted_df = df_with_order.orderBy("number", "sys_updated_at", "sort_order")

# Drop the helper column if necessary
sorted_df = sorted_df.drop("sort_order")

In [ ]:
sorted_df.toPandas()

### Handling Null Values

In [ ]:
# replace question marks with Null
sorted_df = sorted_df.na.replace("?", None)

In [ ]:
total_rows = sorted_df.count()

null_count = sorted_df.select([(F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c) for c in sorted_df.columns])

# Convert the result to a Pandas DataFrame
null_count_pd = null_count.toPandas()

# Filter columns with null counts greater than 0
null_count_pd.loc[:, null_count_pd.iloc[0] > 0]

In [ ]:
# columns that having more than 90% of null values are removed
null_removed = sorted_df.drop("problem_id", "rfc", "vendor","caused_by","cmdb_ci")

In [ ]:
cleaned = null_removed.fillna('Unknown')

In [ ]:
total_rows = sorted_df.count()

null_count = cleaned.select([(F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c) for c in null_removed.columns])

null_count_pd = null_count.toPandas()

null_count_pd.loc[:, null_count_pd.iloc[0] > 0]

### Type Casting

In [ ]:
cleaned.printSchema()

In [ ]:
from pyspark.sql.functions import col, to_timestamp, to_date, year, month
 
# Cast columns to original data types
final = cleaned \
    .withColumn("active", col("active").cast("boolean")) \
    .withColumn("reassignment_count", col("reassignment_count").cast("integer")) \
    .withColumn("reopen_count", col("reopen_count").cast("integer")) \
    .withColumn("sys_mod_count", col("sys_mod_count").cast("integer")) \
    .withColumn("made_sla", col("made_sla").cast("boolean")) \
    .withColumn("knowledge", col("knowledge").cast("boolean")) \
    .withColumn("u_priority_confirmation", col("u_priority_confirmation").cast("boolean")) \
    .withColumn("opened_at", to_timestamp(col("opened_at"), "dd/MM/yyyy HH:mm")) \
    .withColumn("sys_created_at", to_timestamp(col("sys_created_at"), "dd/MM/yyyy HH:mm")) \
    .withColumn("sys_updated_at", to_timestamp(col("sys_updated_at"), "dd/MM/yyyy HH:mm")) \
    .withColumn("resolved_at", to_timestamp(col("resolved_at"), "dd/MM/yyyy HH:mm")) \
    .withColumn("closed_at", to_timestamp(col("closed_at"), "dd/MM/yyyy HH:mm"))
 

In [ ]:
final.printSchema()

In [ ]:
# add opened month, opened year, closed month, closed year columns

from pyspark.sql.functions import month, year

final = final \
    .withColumn("opened_month", month(to_timestamp(col("opened_at"), "dd/MM/yyyy HH:mm")))\
    .withColumn("opened_year", year(to_timestamp(col("opened_at"), "dd/MM/yyyy HH:mm")))\
    .withColumn("closed_month", month(to_timestamp(col("closed_at"), "dd/MM/yyyy HH:mm")))\
    .withColumn("closed_year", year(to_timestamp(col("closed_at"), "dd/MM/yyyy HH:mm")))
 

In [ ]:
final.withColumn("time_taken", col('closed_at') - col('opened_at')).select('time_taken').show(truncate=False)

In [ ]:
# add time taken to close an open incident in seconds

from pyspark.sql.functions import from_unixtime, unix_timestamp

final2 = final.withColumn("time_taken", col('closed_at') - col('opened_at'))
final2 = final2.withColumn("total_seconds", expr("unix_timestamp(closed_at) - unix_timestamp(opened_at)"))
final2.select("total_seconds").show(truncate=False)

In [ ]:
final2.printSchema()

In [ ]:
total_rows = sorted_df.count()

null_count = final.select([(F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c) for c in final.columns])

null_count_pd = null_count.toPandas()

null_count_pd.loc[:, null_count_pd.iloc[0] > 0]

In [ ]:
final2.toPandas()

In [ ]:
final2.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("wasbs://capstonetransformed@ustcapstonestorage.blob.core.windows.net/transformed.parquet")